In [1]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx
# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each (or use len(...) for "all")
num_topics = len(topic_cols)  # e.g., 100
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [4]:
# Define your grid
lambda_values = np.random.uniform(0.001, 0.01, 10)
param_grid = {
    'window_sizes': [150, 200, 250, 300],
    'n_lags': [1,2,3,4,5,6,7,8],
    'lambdas': lambda_values
}

# Run grid search 
summary_df, coefficients_df = grid_search(X, y, param_grid, verbose=True)


Testing 320 configurations...


Grid search:  20%|██        | 64/320 [01:00<05:08,  1.20s/it]/Users/tommasodifrancesco/Desktop/Lasso_paper/Empirical/scripts/lasso_11_2025/stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept
Grid search:  22%|██▎       | 72/320 [01:10<04:59,  1.21s/it]/Users/tommasodifrancesco/Desktop/Lasso_paper/Empirical/scripts/lasso_11_2025/stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept
Grid search:  90%|█████████ | 288/320 [07:21<00:55,  1.73s/it]/Users/tommasodifrancesco/Desktop/Lasso_paper/Empirical/scripts/lasso_11_2025/stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept
Grid search:  92%|█████████▎| 296/320 [07:40<00:46,  1.92s/it]/Users/tommasodifrancesco/Desktop/Lasso_paper/Emp


GRID SEARCH COMPLETE


In [10]:
summary_df.to_parquet('grid_search_summary_8.parquet', index=False)


In [6]:
coefficients_df.to_parquet('grid_search_coefficients_8.parquet', index=False)